In [49]:
import torch

In [50]:
def hb_transform(x):
    if len(x.shape) == 1:
        x = x.view(1,-1)
    (m,n) = x.shape
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.sum(1+i,keepdim=True) - 2*x
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,) + tuple(2*i+1 for i in range(k)) + tuple(2*i+2 for i in range(k)))
    return x.reshape(m, 2**k, 2**k)

In [51]:
mats = hb_transform(torch.eye(16))
for i in range(len(mats)):
    for j in range(len(mats)):
        if i != j:
            try:
                torch.testing.assert_close(mats[i] @ mats[j].T, -1 * (mats[j] @ mats[i].T))
                print(f"{i} and {j} anti-commute")
            except AssertionError:
                pass

0 and 3 anti-commute
0 and 7 anti-commute
0 and 11 anti-commute
0 and 12 anti-commute
0 and 13 anti-commute
0 and 14 anti-commute
1 and 2 anti-commute
1 and 6 anti-commute
1 and 10 anti-commute
1 and 12 anti-commute
1 and 13 anti-commute
1 and 15 anti-commute
2 and 1 anti-commute
2 and 5 anti-commute
2 and 9 anti-commute
2 and 12 anti-commute
2 and 14 anti-commute
2 and 15 anti-commute
3 and 0 anti-commute
3 and 4 anti-commute
3 and 8 anti-commute
3 and 13 anti-commute
3 and 14 anti-commute
3 and 15 anti-commute
4 and 3 anti-commute
4 and 7 anti-commute
4 and 8 anti-commute
4 and 9 anti-commute
4 and 10 anti-commute
4 and 15 anti-commute
5 and 2 anti-commute
5 and 6 anti-commute
5 and 8 anti-commute
5 and 9 anti-commute
5 and 11 anti-commute
5 and 14 anti-commute
6 and 1 anti-commute
6 and 5 anti-commute
6 and 8 anti-commute
6 and 10 anti-commute
6 and 11 anti-commute
6 and 13 anti-commute
7 and 0 anti-commute
7 and 4 anti-commute
7 and 9 anti-commute
7 and 10 anti-commute
7 and 11 ant

In [52]:
num_mats = len(mats)
k = 3
adjacency_matrix = torch.zeros((num_mats, num_mats), dtype=torch.bool)

for i in range(num_mats):
    for j in range(i+1, num_mats):
        if torch.allclose(mats[i] @ mats[j].T, -1 * (mats[j] @ mats[i].T)):
            adjacency_matrix[i,j] = True
            adjacency_matrix[j,i] = True
    
walk = [0]
visited = set([0])
to_visit = [[0, 0]]

while len(to_visit) > 0:
    if len(walk) == num_mats:
        break
    curr, next = to_visit[-1]
    next_found = False

    for next_node in range(next, num_mats):
        if next_node not in visited:
            start = 0 if len(walk) - k + 1 < 0 else len(walk) - k + 1
            anti_commutes = True
            for i in range(start, len(walk)):
                if not adjacency_matrix[walk[i], next_node]:
                    anti_commutes = False
                    break
            if anti_commutes:
                to_visit[-1][1] = next_node + 1
                to_visit.append([next_node, 0])
                walk.append(next_node)
                visited.add(next_node)
                next_found = True
                break
    
    if not next_found:
        removed, _ = to_visit.pop()
        walk.pop()
        visited.remove(removed)
        if len(to_visit) == 0 and removed < num_mats - 1:
            to_visit.append([removed + 1, 0])
            walk.append(removed + 1)
            visited.add(removed + 1)

if len(walk) == num_mats:
    print(walk)
else:
    print("No valid permutation")

No valid permutation


In [53]:
def hb_transform(x):
    if len(x.shape) == 1:
        x = x.view(1,-1)
    (m,n) = x.shape
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    b = torch.tensor([1,1,-1,-1]).to(x.dtype)
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.flip(1+i) + x * b.view((1,)*(i+1) + (4,) + (1,)*(k-1-i))
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,) + tuple(2*i+1 for i in range(k)) + tuple(2*i+2 for i in range(k)))
    return x.reshape(m, 2**k, 2**k)

In [54]:
mats = hb_transform(torch.eye(64))
for i in range(len(mats)):
    for j in range(len(mats)):
        if i != j:
            try:
                torch.testing.assert_close(mats[i] @ mats[j].T, -1 * (mats[j] @ mats[i].T))
                print(f"{i} and {j} anti-commute")
            except AssertionError:
                pass

0 and 2 anti-commute
0 and 6 anti-commute
0 and 8 anti-commute
0 and 9 anti-commute
0 and 11 anti-commute
0 and 14 anti-commute
0 and 18 anti-commute
0 and 22 anti-commute
0 and 24 anti-commute
0 and 25 anti-commute
0 and 27 anti-commute
0 and 30 anti-commute
0 and 32 anti-commute
0 and 33 anti-commute
0 and 35 anti-commute
0 and 36 anti-commute
0 and 37 anti-commute
0 and 39 anti-commute
0 and 42 anti-commute
0 and 44 anti-commute
0 and 45 anti-commute
0 and 47 anti-commute
0 and 50 anti-commute
0 and 54 anti-commute
0 and 56 anti-commute
0 and 57 anti-commute
0 and 59 anti-commute
0 and 62 anti-commute
1 and 3 anti-commute
1 and 7 anti-commute
1 and 8 anti-commute
1 and 9 anti-commute
1 and 10 anti-commute
1 and 15 anti-commute
1 and 19 anti-commute
1 and 23 anti-commute
1 and 24 anti-commute
1 and 25 anti-commute
1 and 26 anti-commute
1 and 31 anti-commute
1 and 32 anti-commute
1 and 33 anti-commute
1 and 34 anti-commute
1 and 36 anti-commute
1 and 37 anti-commute
1 and 38 anti-comm

In [55]:
num_mats = len(mats)
k = 2
adjacency_matrix = torch.zeros((num_mats, num_mats), dtype=torch.bool)

for i in range(num_mats):
    for j in range(i+1, num_mats):
        if torch.allclose(mats[i] @ mats[j].T, -1 * (mats[j] @ mats[i].T)):
            adjacency_matrix[i,j] = True
            adjacency_matrix[j,i] = True
    
walk = [0]
visited = set([0])
to_visit = [[0, 0]]

while len(to_visit) > 0:
    if len(walk) == num_mats:
        break
    curr, next = to_visit[-1]
    next_found = False

    for next_node in range(next, num_mats):
        if next_node not in visited:
            start = 0 if len(walk) - k + 1 < 0 else len(walk) - k + 1
            anti_commutes = True
            for i in range(start, len(walk)):
                if not adjacency_matrix[walk[i], next_node]:
                    anti_commutes = False
                    break
            if anti_commutes:
                to_visit[-1][1] = next_node + 1
                to_visit.append([next_node, 0])
                walk.append(next_node)
                visited.add(next_node)
                next_found = True
                break
    
    if not next_found:
        removed, _ = to_visit.pop()
        walk.pop()
        visited.remove(removed)
        if len(to_visit) == 0 and removed < num_mats - 1:
            to_visit.append([removed + 1, 0])
            walk.append(removed + 1)
            visited.add(removed + 1)

if len(walk) == num_mats:
    print(walk)
else:
    print("No valid permutation")

[0, 2, 4, 6, 8, 1, 3, 5, 7, 9, 11, 13, 15, 17, 10, 12, 14, 16, 18, 20, 22, 24, 19, 21, 23, 25, 27, 29, 31, 33, 26, 28, 30, 32, 34, 36, 38, 40, 35, 37, 39, 41, 43, 45, 47, 49, 42, 44, 46, 48, 50, 52, 54, 56, 51, 53, 59, 61, 63, 57, 55, 62, 60, 58]


In [58]:
# bron-kerbosch algorithm

adjacency_list = {}

for i in range(num_mats):
    adjacency_list[i] = []

for i in range(num_mats):
    for j in range(i+1, num_mats):
        if torch.allclose(mats[i] @ mats[j].T, -1 * (mats[j] @ mats[i].T)):
            adjacency_list[i].append(j)
            adjacency_list[j].append(i)

def bron_kerbosch(r, p, x, cliques):
    if not p and not x:
        cliques.append(list(r))
        return
    pivot = p.union(x).pop()
    p.union(x).add(pivot)
    for node in p.union(x):
        if len(adjacency_list[node]) > len(adjacency_list[pivot]):
            pivot = node
    rest = p.difference(adjacency_list[pivot])
    for node in list(rest):
        neighbors = adjacency_list[node]
        bron_kerbosch(r.union([node]), p.intersection(neighbors), x.intersection(neighbors), cliques)
        p.remove(node)
        x.add(node)


all_cliques = []
r = set()
p = set(range(num_mats))
x = set()

bron_kerbosch(r, p, x, all_cliques)

cliques_8 = []
for clique in all_cliques:
    if len(clique) == 8:
        cliques_8.append(clique)

unique_cliques = []
visited = set()

for clique in cliques_8:
    in_visited = False
    for node in clique:
        if node in visited:
            in_visited = True
            break
    if not in_visited:
        unique_cliques.append(clique)
        for node in clique:
            visited.add(node)
    if len(unique_cliques) == 8:
        break

for clique in unique_cliques:
    print(sorted(clique))

[0, 2, 11, 25, 33, 39, 47, 57]
[1, 3, 10, 24, 32, 38, 46, 56]
[4, 6, 15, 29, 35, 37, 43, 61]
[5, 7, 14, 28, 34, 36, 42, 60]
[12, 18, 20, 26, 44, 53, 55, 62]
[13, 19, 21, 27, 45, 52, 54, 63]
[8, 16, 22, 30, 40, 49, 51, 58]
[9, 17, 23, 31, 41, 48, 50, 59]
